# Brain Tumor MRI — Colab (one cell)

**Runtime → Change runtime type → GPU** (recommended).

Open the code cell below, edit **EPOCHS**, **MODELS**, and data options, then run that single cell. It mounts Drive, clones the repo, installs dependencies, prepares data, runs training, and prints **final_results.csv** at the end.

### Google Drive dataset

To train on files already in Drive:

1. Set **`DATA_SOURCE = "drive"`**.
2. Set **`DRIVE_DATA_PATH`** to the **full path after mount** — the folder that contains **`Training/`** (and **`Testing/`** if present).
   - Personal Drive: `/content/drive/MyDrive/your_folder`
   - Shared drive: `/content/drive/Shareddrives/<DriveName>/your_folder`
3. **`DRIVE_USE_IN_PLACE`**: `False` copies data into `data/raw/` (often faster). `True` passes **`--data_dir`** so preprocessing reads directly from Drive (saves Colab disk; can be slower).

### Kaggle dataset

Set **`DATA_SOURCE = "kaggle"`**. Put **`kaggle.json`** on Drive at **`KAGGLE_JSON_DRIVE`**, or upload when prompted.


In [ ]:
# =============================================================================
# EDIT THESE — then run this cell (Runtime → Change runtime type → GPU recommended)
# =============================================================================
EPOCHS = 2
# One model or several (space-separated): resnet50 | vit | hybrid
MODELS = "resnet50"

REPO_URL = "https://github.com/MuhammadSaljooq/Tumor-AI-training-model.git"
BRANCH = "checkpoint_added"
PROJECT_DIR = "/content/Tumor-AI-training-model"
CONFIG_FILE = "configs/config_colab.yaml"
# Optional extra CLI args, e.g. '--resume results/checkpoints/resnet50_last.pth'
EXTRA_ARGS = ""

# --- Data source ---
# "kaggle" — download dataset (needs kaggle.json on Drive or upload).
# "drive" — use a folder on your mounted Google Drive (see below).
DATA_SOURCE = "kaggle"

# --- Google Drive dataset (when DATA_SOURCE == "drive") ---
# After drive.mount("/content/drive"), use the full path to the folder that contains
#   Training/  (and Testing/ if you have it).
# Examples:
#   My Drive:     "/content/drive/MyDrive/brain_tumor_data"
#   Shared drive: "/content/drive/Shareddrives/MyTeamDataset/brain_tumor_data"
DRIVE_DATA_PATH = "/content/drive/MyDrive/brain_tumor_data"
# True  — read raw images directly from Drive (no copy). Saves disk; can be slower / I/O heavy.
# False — copy dataset into the repo under data/raw/ (often faster training).
DRIVE_USE_IN_PLACE = False

# --- Kaggle API token (when DATA_SOURCE == "kaggle") ---
# Path to kaggle.json on Drive (skips upload), or None to always use the upload widget.
KAGGLE_JSON_DRIVE = "/content/drive/MyDrive/kaggle.json"
# =============================================================================

import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run(cmd, cwd=None):
    print("+", cmd if isinstance(cmd, str) else " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd)


from google.colab import drive

drive.mount("/content/drive")
subprocess.run(["nvidia-smi"], check=False)

if Path(PROJECT_DIR).exists():
    print("Updating repo...")
    run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH])
    run(["git", "-C", PROJECT_DIR, "checkout", BRANCH])
    run(["git", "-C", PROJECT_DIR, "pull", "origin", BRANCH])
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
if DATA_SOURCE == "kaggle":
    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
print("Project:", PROJECT_DIR)

data_dir_for_main = None

if DATA_SOURCE == "kaggle":
    raw = Path(PROJECT_DIR) / "data" / "raw"
    raw.mkdir(parents=True, exist_ok=True)

    kdir = Path.home() / ".kaggle"
    kdir.mkdir(parents=True, exist_ok=True)
    kaggle_dest = kdir / "kaggle.json"
    got = False
    if KAGGLE_JSON_DRIVE:
        src_k = Path(KAGGLE_JSON_DRIVE)
        if src_k.is_file():
            shutil.copy2(src_k, kaggle_dest)
            os.chmod(kaggle_dest, 0o600)
            print("Using kaggle.json from Drive:", src_k)
            got = True
    if not got:
        from google.colab import files

        print("Upload kaggle.json (Kaggle → Account → API → Create Token)")
        uploaded = files.upload()
        for name, data in uploaded.items():
            if name.endswith(".json"):
                kaggle_dest.write_bytes(data)
                os.chmod(kaggle_dest, 0o600)
                got = True
                break
        if not got:
            raise RuntimeError("Upload kaggle.json")

    data_parent = Path(PROJECT_DIR) / "data"
    zip_path = data_parent / "brain-tumor-mri-dataset.zip"
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            "masoudnickparvar/brain-tumor-mri-dataset",
            "-p",
            str(data_parent),
        ],
        check=True,
        cwd=PROJECT_DIR,
    )
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(raw)
    zip_path.unlink(missing_ok=True)

elif DATA_SOURCE == "drive":
    drive_root = Path(DRIVE_DATA_PATH).expanduser().resolve()
    if not drive_root.is_dir():
        raise FileNotFoundError(
            f"Not a directory: {drive_root}\n"
            "Use the path after mounting Drive, e.g. /content/drive/MyDrive/... or "
            "/content/drive/Shareddrives/<Name>/..."
        )

    if DRIVE_USE_IN_PLACE:
        subs_d = [
            p
            for p in drive_root.iterdir()
            if p.is_dir() and p.name not in {".ipynb_checkpoints"}
        ]
        if (
            len(subs_d) == 1
            and (subs_d[0] / "Training").is_dir()
            and not (drive_root / "Training").is_dir()
        ):
            effective_raw = subs_d[0]
            print("Using nested folder as raw data root:", effective_raw)
        else:
            effective_raw = drive_root
        if not (effective_raw / "Training").is_dir():
            raise FileNotFoundError(
                f"Expected a Training/ folder under {effective_raw}. "
                "Point DRIVE_DATA_PATH at the folder that contains Training/ (or one level above if a single nested folder holds it)."
            )
        data_dir_for_main = str(effective_raw)
        raw = effective_raw
        print("Drive in-place raw dir (--data_dir):", data_dir_for_main)
    else:
        raw = Path(PROJECT_DIR) / "data" / "raw"
        raw.mkdir(parents=True, exist_ok=True)
        for item in drive_root.iterdir():
            dest = raw / item.name
            if dest.exists():
                if dest.is_dir():
                    shutil.rmtree(dest)
                else:
                    dest.unlink()
            if item.is_dir():
                shutil.copytree(item, dest)
            else:
                shutil.copy2(item, dest)
else:
    raise ValueError('DATA_SOURCE must be "kaggle" or "drive"')

if DATA_SOURCE == "kaggle" or (DATA_SOURCE == "drive" and not DRIVE_USE_IN_PLACE):
    subs = [p for p in raw.iterdir() if p.is_dir() and p.name not in {".ipynb_checkpoints"}]
    if len(subs) == 1 and (subs[0] / "Training").is_dir():
        inner = subs[0]
        for item in inner.iterdir():
            shutil.move(str(item), str(raw / item.name))
        inner.rmdir()
        print("Flattened:", raw)

if (raw / "Training").is_dir():
    print("OK: Training/ found at", raw / "Training")
else:
    print("WARNING: expected Training/ under", raw)

import shlex

model_list = MODELS.split()
cmd = [
    sys.executable,
    "main.py",
    "--config",
    CONFIG_FILE,
    "--models",
    *model_list,
    "--epochs",
    str(int(EPOCHS)),
]
if data_dir_for_main:
    cmd += ["--data_dir", data_dir_for_main]
if EXTRA_ARGS.strip():
    cmd += shlex.split(EXTRA_ARGS.strip())

os.chdir(PROJECT_DIR)
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=PROJECT_DIR)

results_dir = Path(PROJECT_DIR) / "results"
csv_path = results_dir / "final_results.csv"
if csv_path.is_file():
    print("\n--- final_results.csv ---\n")
    print(csv_path.read_text())
print("\nDone. Artifacts: results/checkpoints/, results/plots/, results/final_results.csv")

